# Jargonaut: BM25 retrieval and Recall@K evaluation

Stage 1 of the pipeline. For a contract and a question, BM25 ranks the contract's chunks and the top 10
go to DeBERTa. This notebook measures how often the gold evidence reaches that top 10.

| Step | What it does | Split |
|---|---|---|
| 1 | One question, one contract: see the ranking | val |
| 2 | Baseline Recall@K over all answerable questions | val |
| 3 | Compare query forms (full template / name + description / name only) | val |
| 4 | Tune preprocessing (stopwords, stemming) and k1/b | val |
| 5 | Final evaluation, setups fixed in advance, run once | **test** |
| 6 | Error analysis: why gold spans are missed | test |

**Data:** the shared `Data Prep.zip` (split, `qa.jsonl.gz`, chunks `chunks_words_m300_o60`, gold chunk labels).
Nothing here changes the shared data. Everything is deterministic, so rerunning gives the same numbers.

**Runs in Colab or locally.** In Colab it unzips `Data Prep.zip` from the group Drive folder. Locally, run it from
the folder that contains `data_prep/`. No GPU needed; the whole notebook takes about a minute.

## 0. Setup

In [20]:
import importlib.util, subprocess, sys
for pkg, mod in [("rank_bm25", "rank_bm25"), ("nltk", "nltk"), ("pandas", "pandas")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [21]:
import gzip, json, re, zipfile
from functools import lru_cache
from pathlib import Path

import pandas as pd
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_PATH = Path("/content/drive/MyDrive/CS425 Grp Project/EvidenceExtraction/Data Prep.zip")
    assert ZIP_PATH.is_file(), f"Missing ZIP: {ZIP_PATH}"
    if not Path("/content/data_prep").exists():
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall("/content")
    DATA_ROOT = Path("/content/data_prep")
    OUT_DIR = Path("/content/drive/MyDrive/CS425 Grp Project/BM25/outputs")
else:
    # works whether the notebook sits next to data_prep/ or inside it
    candidates = [Path("data_prep"), Path("."), Path("..") / "data_prep"]
    DATA_ROOT = next((p for p in candidates if (p / "outputs" / "processed").is_dir()), None)
    assert DATA_ROOT is not None, (f"Can't find data_prep/outputs from {Path.cwd()}. "
                                   "Put this notebook next to or inside the data_prep folder.")
    OUT_DIR = DATA_ROOT / "outputs" / "bm25"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED = DATA_ROOT / "outputs" / "processed"
CHUNK_DIR = DATA_ROOT / "outputs" / "chunks" / "chunks_words_m300_o60"

CATEGORIES = [   # the 10 proposed in data_prep/README.md, section 1
    "Governing Law", "Expiration Date", "Renewal Term", "Termination For Convenience",
    "Cap On Liability", "Anti-Assignment", "Non-Compete", "Exclusivity",
    "Ip Ownership Assignment", "Insurance",
]
KS = [1, 3, 5, 10, 20]
K = 10                      # operating point: chunks passed to DeBERTa
INF = float("inf")
print("Data:", DATA_ROOT.resolve(), "\nOutputs:", OUT_DIR.resolve())

Data: /Users/ridheemasharma/Downloads/data_prep 
Outputs: /Users/ridheemasharma/Downloads/data_prep/outputs/bm25


### Load the shared data
- `qa`: one row per (contract, question), with all gold answers. Only our 10 categories.
- `chunks`: 300-word windows with 60-word overlap, each with its character offsets in the contract.
- `gold`: for every gold span, which chunks fully contain it (**strict**) or overlap it (**lenient**).

In [22]:
def read_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return pd.DataFrame([json.loads(line) for line in f if line.strip()])

contracts = read_jsonl_gz(PROCESSED / "contracts.jsonl.gz").set_index("contract_id")["context"]
qa = read_jsonl_gz(PROCESSED / "qa.jsonl.gz")
qa = qa[qa["category"].isin(CATEGORIES)].reset_index(drop=True)
chunks = read_jsonl_gz(CHUNK_DIR / "chunks.jsonl.gz")
gold = read_jsonl_gz(CHUNK_DIR / "gold_chunk_labels.jsonl.gz").set_index("question_id")

assert set(CATEGORIES) <= set(qa["category"]), "A category name doesn't match the data"
chunks_by_contract = {cid: cc.sort_values("chunk_index").reset_index(drop=True)
                      for cid, cc in chunks.groupby("contract_id")}

summary = qa[qa["split"].isin(["train", "val", "test"])].groupby("split").agg(
    contracts=("contract_id", "nunique"), questions=("question_id", "size"),
    answerable=("has_answer", "sum"))
summary

,contracts,questions,answerable
split,,,
test,102,1020,443
train,346,3460,1713
val,61,610,287


### Shared building blocks
Every step below reuses these, so the steps differ only in the settings they pass in.

- **Tokenisers.** `raw`: lowercase + keep runs of letters/digits. `final`: also removes stopwords
  (NLTK English list, **keeping** *no, nor, not, without*, because in contracts negation is the meaning) and applies
  Porter stemming. Chunks and queries always go through the **same** tokeniser.
- **Query forms.** Every CUAD question is
  `Highlight the parts (if any) of this contract related to "NAME" that should be reviewed by a lawyer. Details: DESCRIPTION`.
  `A` = the whole thing, `B` = NAME + DESCRIPTION, `C` = NAME only.
- **`rank_spans`.** Builds one BM25 index per contract (IDF is computed over that contract's chunks only, as in the app),
  scores every question, and records each gold span's **best rank**: the rank of the highest-ranked chunk containing it
  (infinity if no chunk contains it). "Found at K" is then just `best rank <= K`.
- **Metrics** (macro average over answerable questions):
  - **Recall@K** = share of a question's gold spans found in the top K
  - **Hit@K** = 1 if at least one gold span is in the top K

In [23]:
NLTK_STOPWORDS = '''i me my myself we our ours ourselves you your yours yourself yourselves he him his himself
she her hers herself it its itself they them their theirs themselves what which who whom this that these those
am is are was were be been being have has had having do does did doing a an the and but if or because as until
while of at by for with about against between into through during before after above below to from up down in
out on off over under again further then once here there when where why how all any both each few more most
other some such only own same so than too very s t can will just don should now d ll m o re ve y'''.split()
STOPWORDS = set(NLTK_STOPWORDS) - {"no", "nor", "not", "without"}
_stemmer = PorterStemmer()

@lru_cache(maxsize=None)
def stem(word):
    return _stemmer.stem(word)

def make_tokenizer(remove_stopwords=False, use_stemming=False):
    def tokenize(text):
        words = re.findall(r"[a-z0-9]+", text.lower())
        if remove_stopwords:
            words = [w for w in words if w not in STOPWORDS]
        if use_stemming:
            words = [stem(w) for w in words]
        return words
    return tokenize

tok_raw = make_tokenizer()
tok_final = make_tokenizer(remove_stopwords=True, use_stemming=True)

TEMPLATE = re.compile(r'related to "(.+?)" that should be reviewed by a lawyer\.\s*Details:\s*(.*)', re.S)
def slots(question):
    m = TEMPLATE.search(question)
    if m is None:
        raise ValueError(f"Not a CUAD template question: {question[:80]}")
    return m.groups()

QUERY_FORMS = {
    "A": lambda q: q,                      # full CUAD template
    "B": lambda q: " ".join(slots(q)),     # name + description
    "C": lambda q: slots(q)[0],            # name only
}
assert qa["question"].map(lambda q: TEMPLATE.search(q) is not None).all()


def rank_spans(split, query_form="B", tokenize=tok_final, k1=1.5, b=0.75):
    '''One row per gold span of every answerable question in `split`.'''
    questions = qa[(qa["split"] == split) & qa["has_answer"]]
    make_query = QUERY_FORMS[query_form]
    indexes = {}
    rows = []
    for q in questions.itertuples():
        if q.contract_id not in indexes:
            cc = chunks_by_contract[q.contract_id]
            indexes[q.contract_id] = (cc, BM25Okapi([tokenize(t) for t in cc["text"]], k1=k1, b=b))
        cc, bm25 = indexes[q.contract_id]
        query = make_query(q.question)
        scores = bm25.get_scores(tokenize(query))
        order = sorted(range(len(cc)), key=lambda i: -scores[i])
        rank_of = {cc.chunk_id[i]: r + 1 for r, i in enumerate(order)}
        for j, s in enumerate(gold.loc[q.question_id, "spans"]):
            rows.append({
                "question_id": q.question_id, "contract_id": q.contract_id, "category": q.category,
                "span": j, "answer_start": s["answer_start"], "answer_end": s["answer_end"], "query": query,
                "best_strict": min((rank_of[c] for c in s["strict_chunk_ids"]), default=INF),
                "best_lenient": min((rank_of[c] for c in s["lenient_chunk_ids"]), default=INF),
                "top1_chunk_index": int(cc.chunk_index[order[0]]),
            })
    return pd.DataFrame(rows)


def per_question(spans):
    '''Recall@K and Hit@K per question, strict and lenient.'''
    out = {}
    for k in KS:
        for mode in ["strict", "lenient"]:
            found = spans[f"best_{mode}"] <= k
            g = found.groupby(spans["question_id"])
            out[f"recall_{mode}@{k}"] = g.mean()
            out[f"hit_{mode}@{k}"] = g.max().astype(float)
    res = pd.DataFrame(out)
    res["category"] = spans.groupby("question_id")["category"].first()
    return res


def metric_table(res, metrics=("recall_strict", "recall_lenient", "hit_strict", "hit_lenient")):
    return pd.DataFrame({m: [res[f"{m}@{k}"].mean() for k in KS] for m in metrics},
                        index=[f"K={k}" for k in KS]).round(3)

## Step 1: one question, one contract
Sanity check of the whole flow before averaging anything. Raw tokeniser and the full template (form A),
which is the starting point.

In [24]:
q = qa[(qa["split"] == "val") & qa["has_answer"] & (qa["category"] == "Termination For Convenience")].iloc[0]
cc = chunks_by_contract[q.contract_id]
bm25 = BM25Okapi([tok_raw(t) for t in cc["text"]])          # library defaults: k1=1.5, b=0.75
ranked = cc.assign(score=bm25.get_scores(tok_raw(q.question))).sort_values("score", ascending=False)
gold_ids = set(gold.loc[q.question_id, "strict_chunk_ids"])

print(f"{q.contract_id}: {len(cc)} chunks\nQuestion: {q.question}\n")
for rank, row in enumerate(ranked.head(10).itertuples(), 1):
    print(f"{rank:2d}. {row.chunk_id}  score={row.score:6.2f}{'  <-- GOLD' if row.chunk_id in gold_ids else ''}")

cuad_053: 39 chunks
Question: Highlight the parts (if any) of this contract related to "Termination For Convenience" that should be reviewed by a lawyer. Details: Can a party terminate this  contract without cause (solely by giving a notice and allowing a waiting  period to expire)?

 1. cuad_053_c0012  score= 28.99  <-- GOLD
 2. cuad_053_c0000  score= 28.24
 3. cuad_053_c0017  score= 28.06
 4. cuad_053_c0015  score= 26.09
 5. cuad_053_c0035  score= 25.97
 6. cuad_053_c0013  score= 25.58
 7. cuad_053_c0016  score= 25.34
 8. cuad_053_c0020  score= 24.81
 9. cuad_053_c0002  score= 24.31
10. cuad_053_c0009  score= 23.75


**Reading it:** the gold chunk is rank 1, but only just (28.99 vs 28.24 for the title page). Breaking the score
down word by word showed why: stopwords add roughly the same ~11 points to every chunk, template words like
"contract", "should" and "reviewed" are rare in this contract and so get high IDF, and the category name word
"convenience" does not appear in the contract at all. One question proves the pipeline works, not that BM25 does.

## Step 2: baseline Recall@K on validation
Same as step 1 for every answerable validation question. Questions with no gold span are excluded: recall is 0/0 for
them, and deciding "nothing here" is DeBERTa's job (answerability threshold).

In [25]:
base_val = per_question(rank_spans("val", query_form="A", tokenize=tok_raw))
print(f"{len(base_val)} answerable val questions\n")
metric_table(base_val)

287 answerable val questions



,recall_strict,recall_lenient,hit_strict,hit_lenient
K=1,0.418,0.462,0.474,0.519
K=3,0.595,0.649,0.652,0.697
K=5,0.656,0.714,0.718,0.767
K=10,0.798,0.859,0.854,0.899
K=20,0.863,0.911,0.899,0.937


In [26]:
(base_val.groupby("category")
    .agg(questions=("recall_strict@10", "size"), recall_strict_at_10=("recall_strict@10", "mean"),
         hit_strict_at_10=("hit_strict@10", "mean"))
    .round(3).sort_values("recall_strict_at_10"))

,questions,recall_strict_at_10,hit_strict_at_10
category,,,
Non-Compete,11,0.370,0.636
Exclusivity,17,0.441,0.588
Ip Ownership Assignment,18,0.456,0.556
Cap On Liability,34,0.741,0.853
Insurance,21,0.808,0.857
Termination For Convenience,21,0.810,0.810
Expiration Date,48,0.858,0.896
Anti-Assignment,43,0.905,0.953
Governing Law,54,0.944,0.944


**Reading it:** about 80% of gold spans reach the top 10. The curve flattens after K=10 (+14 points from 5 to 10,
+6.5 from 10 to 20), which supports K=10. Hit rate is above recall because some questions have several gold spans
and only some are found. Small categories (Non-Compete has 11 questions, so one question = 9 points) are noisy.

## Step 3: query forms (validation)

In [27]:
for form in "ABC":
    print(f"{form}: {QUERY_FORMS[form](q.question)}\n")

forms_val = {form: per_question(rank_spans("val", query_form=form, tokenize=tok_raw)) for form in "ABC"}
pd.DataFrame({f: [r[f"recall_strict@{k}"].mean() for k in KS] for f, r in forms_val.items()},
             index=[f"K={k}" for k in KS]).round(3)

A: Highlight the parts (if any) of this contract related to "Termination For Convenience" that should be reviewed by a lawyer. Details: Can a party terminate this  contract without cause (solely by giving a notice and allowing a waiting  period to expire)?

B: Termination For Convenience Can a party terminate this  contract without cause (solely by giving a notice and allowing a waiting  period to expire)?

C: Termination For Convenience



,A,B,C
K=1,0.418,0.463,0.383
K=3,0.595,0.655,0.570
K=5,0.656,0.714,0.652
K=10,0.798,0.798,0.756
K=20,0.863,0.881,0.849


In [28]:
pd.DataFrame({f: r.groupby("category")["recall_strict@10"].mean() for f, r in forms_val.items()}
             ).assign(questions=base_val.groupby("category").size()).round(3).sort_values("A")

,A,B,C,questions
category,,,,
Non-Compete,0.370,0.274,0.200,11
Exclusivity,0.441,0.390,0.539,17
Ip Ownership Assignment,0.456,0.467,0.528,18
Cap On Liability,0.741,0.783,0.673,34
Insurance,0.808,0.860,0.897,21
Termination For Convenience,0.810,0.857,0.857,21
Expiration Date,0.858,0.816,0.691,48
Anti-Assignment,0.905,0.865,0.752,43
Governing Law,0.944,0.981,0.963,54


**Reading it:** removing the boilerplate (B) pushes gold chunks higher (about +5 points at K=1/3/5) but ties A at
K=10. Name only (C) is worst: the description supplies words that clauses actually use. **B is used from here on.**
Per-category differences under ~10 points are within noise at these sample sizes.

## Step 4: tuning on validation
One change at a time, so each effect can be attributed. Then a k1/b grid on the best preprocessing.

In [29]:
SETTINGS = {
    "0 baseline B":             make_tokenizer(False, False),
    "1 + stopwords":            make_tokenizer(True, False),
    "2 + stemming":             make_tokenizer(False, True),
    "3 + stopwords + stemming": make_tokenizer(True, True),
}
prep_val = {name: per_question(rank_spans("val", "B", tok)) for name, tok in SETTINGS.items()}
pd.DataFrame({n: {f"R@{k}": r[f"recall_strict@{k}"].mean() for k in KS} for n, r in prep_val.items()}).T.round(3)

,R@1,R@3,R@5,R@10,R@20
0 baseline B,0.463,0.655,0.714,0.798,0.881
1 + stopwords,0.468,0.662,0.739,0.797,0.884
2 + stemming,0.500,0.664,0.748,0.818,0.894
3 + stopwords + stemming,0.507,0.661,0.757,0.827,0.882


In [30]:
pd.DataFrame({n: r.groupby("category")["recall_strict@10"].mean() for n, r in prep_val.items()}
             ).assign(questions=base_val.groupby("category").size()).round(3).sort_values("0 baseline B")

,0 baseline B,1 + stopwords,2 + stemming,3 + stopwords + stemming,questions
category,,,,,
Non-Compete,0.274,0.352,0.383,0.400,11
Exclusivity,0.390,0.439,0.458,0.627,17
Ip Ownership Assignment,0.467,0.511,0.539,0.553,18
Cap On Liability,0.783,0.759,0.810,0.821,34
Expiration Date,0.816,0.795,0.806,0.795,48
Termination For Convenience,0.857,0.857,0.810,0.810,21
Insurance,0.860,0.890,0.890,0.897,21
Anti-Assignment,0.865,0.851,0.911,0.893,43
Renewal Term,0.950,0.950,0.950,0.950,20


In [31]:
grid = []
for k1 in [0.9, 1.2, 1.5, 2.0]:
    for b in [0.3, 0.5, 0.75, 0.9]:
        r = per_question(rank_spans("val", "B", tok_final, k1=k1, b=b))
        grid.append({"k1": k1, "b": b, "R@5": r["recall_strict@5"].mean(), "R@10": r["recall_strict@10"].mean()})
grid = pd.DataFrame(grid).round(3)
grid.to_csv(OUT_DIR / "tuning_grid_val.csv", index=False)
print("Strict Recall@10 (rows k1, columns b)")
grid.pivot(index="k1", columns="b", values="R@10")

Strict Recall@10 (rows k1, columns b)


b,0.30,0.50,0.75,0.90
k1,,,,
0.9,0.826,0.826,0.822,0.822
1.2,0.827,0.827,0.827,0.827
1.5,0.827,0.827,0.827,0.827
2.0,0.830,0.830,0.830,0.828


In [32]:
lengths = chunks[chunks["split"] == "val"]["text"].map(lambda t: len(tok_raw(t)))
print("Validation chunk length in tokens:")
lengths.describe(percentiles=[0.25, 0.5, 0.75]).round(1)

Validation chunk length in tokens:


count    2597.0
mean      302.0
std        31.0
min         6.0
25%       303.0
50%       306.0
75%       311.0
max       425.0
Name: text, dtype: float64

**Reading it:** stemming is the change that matters (+2 points), stopwords add a little, both together give
**0.827**. Exclusivity improves most on val (stemming lets "exclusivity" match "exclusive"). The k1/b grid is flat
(0.822 to 0.830, about 2 questions), so we keep the standard **k1=1.5, b=0.75** rather than the luckiest cell.
b barely matters because almost every chunk is ~300 words, so |D|/avgdl is ~1 everywhere.

**Final setup:** query form B + stopwords (negations kept) + Porter stemming + k1=1.5, b=0.75.

## Step 5: final evaluation on test
The three setups below were all fixed **before** this cell was run. We report them; we do not choose among them
using test, and nothing above is changed after seeing these numbers.

In [33]:
FINAL_SETUPS = {
    "A raw":   dict(query_form="A", tokenize=tok_raw),
    "B raw":   dict(query_form="B", tokenize=tok_raw),
    "B final": dict(query_form="B", tokenize=tok_final),
}
test_spans = {name: rank_spans("test", **cfg) for name, cfg in FINAL_SETUPS.items()}
test_res = {name: per_question(s) for name, s in test_spans.items()}
final = test_res["B final"]
print(f"{len(final)} answerable test questions across {test_spans['B final'].contract_id.nunique()} contracts\n")
print("Strict Recall@K by setup")
pd.DataFrame({n: [r[f"recall_strict@{k}"].mean() for k in KS] for n, r in test_res.items()},
             index=[f"K={k}" for k in KS]).round(3)

443 answerable test questions across 92 contracts

Strict Recall@K by setup


,A raw,B raw,B final
K=1,0.405,0.468,0.495
K=3,0.645,0.702,0.721
K=5,0.733,0.766,0.774
K=10,0.818,0.845,0.858
K=20,0.902,0.905,0.921


In [34]:
print("Final setup, all metrics")
metric_table(final)

Final setup, all metrics


,recall_strict,recall_lenient,hit_strict,hit_lenient
K=1,0.495,0.560,0.564,0.614
K=3,0.721,0.777,0.779,0.831
K=5,0.774,0.831,0.833,0.883
K=10,0.858,0.900,0.901,0.935
K=20,0.921,0.957,0.955,0.982


In [35]:
per_cat_test = pd.DataFrame({n: r.groupby("category")["recall_strict@10"].mean() for n, r in test_res.items()})
per_cat_test.insert(0, "questions", final.groupby("category").size())
for name, r in test_res.items():
    r.to_csv(OUT_DIR / f"test_per_question_{name.replace(' ', '_')}.csv")
per_cat_test.round(3).sort_values("B final")

,questions,A raw,B raw,B final
category,,,,
Exclusivity,33,0.506,0.536,0.575
Ip Ownership Assignment,23,0.518,0.618,0.616
Non-Compete,23,0.627,0.640,0.716
Cap On Liability,44,0.751,0.782,0.790
Termination For Convenience,29,0.925,0.925,0.902
Expiration Date,78,0.891,0.917,0.904
Renewal Term,26,0.913,0.913,0.913
Insurance,32,0.842,0.929,0.932
Anti-Assignment,72,0.880,0.888,0.933


**Reading it:** A raw < B raw < B final at every K, the same order as on validation. Final strict
**Recall@10 = 0.858** (from 0.818), against a chunking ceiling of 96.6%. Test is a little higher than validation;
that is fine, the setup was never tuned on test. Exclusivity, IP Ownership Assignment, Non-Compete and Cap On
Liability are the weakest four on both splits. Exclusivity's +24 point gain on val shrinks to +7 on test: the val
figure was partly luck from 17 questions.

## Step 6: error analysis (test, final setup)
A gold span is **missed** if its best strict rank is above 10. Each miss is put in one bucket:

| Type | Meaning |
|---|---|
| chunking split | the span is not fully inside any chunk, so strict retrieval can never find it |
| vocabulary mismatch | the span contains under 25% of the query's stems |
| outranked | the span shares the query's words but other chunks scored higher |

plus a separate **near miss** flag for ranks 11-20. **Overlap** = share of the query's unique stems
(after stopwords + stemming) that appear in the gold span.

In [36]:
MISMATCH_THRESHOLD = 0.25
err = test_spans["B final"].copy()
err["span_text"] = [contracts[c][s:e] for c, s, e in zip(err.contract_id, err.answer_start, err.answer_end)]
err["overlap"] = [len(set(tok_final(qr)) & set(tok_final(t))) / len(set(tok_final(qr)))
                  for qr, t in zip(err["query"], err["span_text"])]

def miss_type(r):
    if r.best_strict <= K:
        return "found"
    if r.best_strict == INF:
        return "chunking split"
    return "vocabulary mismatch" if r.overlap < MISMATCH_THRESHOLD else "outranked"

err["type"] = err.apply(miss_type, axis=1)
err["near_miss"] = err["best_strict"].between(K + 1, 20)
missed = err[err["type"] != "found"]
err.drop(columns=["query"]).to_csv(OUT_DIR / "error_analysis_test_spans.csv", index=False)

print(f"{len(err)} gold spans, {len(missed)} missed at K={K} ({100 * len(missed) / len(err):.1f}%)")
print(f"Near misses (rank {K + 1}-20): {missed.near_miss.sum()} ({100 * missed.near_miss.mean():.1f}% of misses)")
print(f"Missed spans whose #1 chunk was the contract's first chunk: {100 * (missed.top1_chunk_index == 0).mean():.1f}%\n")
counts = missed["type"].value_counts()
pd.DataFrame({"spans": counts, "pct_of_misses": (100 * counts / len(missed)).round(1)})

815 gold spans, 167 missed at K=10 (20.5%)
Near misses (rank 11-20): 72 (43.1% of misses)
Missed spans whose #1 chunk was the contract's first chunk: 0.6%



,spans,pct_of_misses
type,,
vocabulary mismatch,119,71.3
chunking split,31,18.6
outranked,17,10.2


In [37]:
ranked_only = err[err["best_strict"] != INF]   # chunking splits are a different problem
print("Average query-stem overlap of the gold span")
print(ranked_only.groupby(ranked_only["best_strict"] <= K)["overlap"].mean()
      .rename({True: "found", False: "missed"}).round(3).to_string(), "\n")
(ranked_only.assign(status=(ranked_only["best_strict"] <= K).map({True: "found", False: "missed"}))
    .pivot_table(index="category", columns="status", values="overlap", aggfunc="mean").round(3))

Average query-stem overlap of the gold span
best_strict
missed    0.104
found     0.270 



status,found,missed
category,,
Anti-Assignment,0.279,0.125
Cap On Liability,0.170,0.080
Exclusivity,0.095,0.076
Expiration Date,0.400,0.250
Governing Law,0.484,0.250
Insurance,0.224,0.208
Ip Ownership Assignment,0.134,0.066
Non-Compete,0.118,0.069
Renewal Term,0.386,NaN


In [38]:
def short(t, n):
    t = " ".join(t.split())
    return t if len(t) <= n else t[:n] + "..."

for cat in ["Exclusivity", "Ip Ownership Assignment", "Non-Compete", "Cap On Liability"]:
    ex = missed[(missed.category == cat) & (missed.type != "chunking split")].sort_values("overlap").head(3)
    print(f"\n=== {cat} ===")
    for r in ex.itertuples():
        top1 = chunks_by_contract[r.contract_id].text[r.top1_chunk_index]
        print(f"[{r.type}] rank {r.best_strict:.0f}, overlap {r.overlap:.2f}")
        print(f"  Gold:        {short(r.span_text, 220)}")
        print(f"  #1 instead:  {short(top1, 140)}")


=== Exclusivity ===
[vocabulary mismatch] rank 12, overlap 0.00
  Gold:        IMPCO expressly agrees that during the existence of this Agreement, IMPCO shall not enter into any other Joint Venture Agreement or Marketing/Distribution Agreement, with any company or person(s) in the Territory with re...
  #1 instead:  the Delhi Courts shall have exclusive jurisdiction over any legal proceedings in respect of this Agreement. The Parties hereby stipulate irr...
[vocabulary mismatch] rank 51, overlap 0.00
  Gold:        SpinCo shall have the sole right to use the title Steam/its generation and use to publish future editions of the Steam Book.
  #1 instead:  of energy sources including, inter alia, nuclear energy sources. SpinCo shall have the sole right to determine the content contained in futu...
[vocabulary mismatch] rank 34, overlap 0.00
  Gold:        As of the Effective Date, there are no rights with respect to the Product or the SIGA Trademarks in the Territory granted by SIGA, in e

**Reading it:**
- **71% of misses are vocabulary mismatch.** Missed spans contain on average 0.10 of the query's stems, found
  spans 0.27. Examples: exclusivity clauses written as prohibitions ("shall not enter into any other ...
  agreement") with no form of "exclusive"; liability caps saying "liable" where the query says "liability",
  which Porter stems differently (*liabl* vs *liabil*).
- **Word senses get confused:** "exclusive jurisdiction" (a governing law clause) outranked a real exclusivity clause.
- **19% are chunking splits** (the EDA shows 1% of spans are over 208 words, and anything over 300 can never fit).
- **Only 10% are truly outranked**; when a clause uses the question's words, BM25 usually finds it.
- **43% of misses are near misses** (rank 11-20): K=20 would recover many, at double the reading cost for DeBERTa.
- The title page was **not** a common false positive (under 1% of misses), which refutes our step 1 guess.
- Some misses are label noise, e.g. a gold span that is just "us.".

**Future work (not done here):** hybrid dense + BM25 retrieval (targets the 71%), clause-based chunking (the 19%),
legal synonym query expansion, and plain-language queries (form D) once the augmented questions exist.

## Step 7: hand-off file for DeBERTa
For **every** question in the 10 categories (train, val, test, answerable or not), save BM25's top 10 chunks with
the **final setup**, in the format agreed with the evidence extraction side. One gzipped JSONL file per split.

Per candidate:
- `contains_gold`: the chunk fully contains at least one gold span (strict)
- `overlaps_gold`: the chunk touches a gold span at all (lenient). `contains_gold=false, overlaps_gold=true` means
  it holds **part** of a clause and must **not** be used as a negative.

**Clean hard negative** = train question, `overlaps_gold == false`.

> **Leakage rule:** the two gold flags are labels, not inputs. On val/test they may only be used to score
> predictions afterwards, never read by the model or the decoder.

In [39]:
TOP_K = 10
FINAL = dict(query_form="B", tokenize=tok_final, k1=1.5, b=0.75)   # frozen in step 5

def export_top_k(split, query_form, tokenize, k1, b, top_k=TOP_K):
    questions = qa[qa["split"] == split]
    make_query = QUERY_FORMS[query_form]
    indexes = {}
    path = OUT_DIR / f"bm25_top{top_k}_{split}.jsonl.gz"
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for q in questions.itertuples():
            if q.contract_id not in indexes:
                cc = chunks_by_contract[q.contract_id]
                indexes[q.contract_id] = (cc, BM25Okapi([tokenize(t) for t in cc["text"]], k1=k1, b=b))
            cc, bm25 = indexes[q.contract_id]
            query = make_query(q.question)
            scores = bm25.get_scores(tokenize(query))
            order = sorted(range(len(cc)), key=lambda i: -scores[i])[:top_k]
            g = gold.loc[q.question_id]
            strict, lenient = set(g["strict_chunk_ids"]), set(g["lenient_chunk_ids"])
            record = {
                "question_id": q.question_id, "contract_id": q.contract_id, "split": split,
                "category": q.category, "has_answer": bool(q.has_answer),
                "question": q.question, "query": query,
                "candidates": [{
                    "chunk_id": cc.chunk_id[i], "rank": r + 1, "bm25_score": round(float(scores[i]), 4),
                    "text": cc.text[i], "char_start": int(cc.char_start[i]), "char_end": int(cc.char_end[i]),
                    "contains_gold": cc.chunk_id[i] in strict, "overlaps_gold": cc.chunk_id[i] in lenient,
                } for r, i in enumerate(order)],
            }
            f.write(json.dumps(record) + "\n")
    return path

export_paths = {split: export_top_k(split, **FINAL) for split in ["train", "val", "test"]}
for split, p in export_paths.items():
    print(f"{split:5s} -> {p}  ({p.stat().st_size / 1e6:.1f} MB)")

train -> outputs/bm25/bm25_top10_train.jsonl.gz  (9.0 MB)
val   -> outputs/bm25/bm25_top10_val.jsonl.gz  (1.9 MB)
test  -> outputs/bm25/bm25_top10_test.jsonl.gz  (2.4 MB)


### Check the export by reading it back
If the files are right, numbers recomputed **only from the files** must match steps 4 and 5 exactly:
- **Hit@10** from `contains_gold` (does any top-10 chunk hold a gold span?)
- **Recall@10** per gold span, by checking each span's strict chunks against the exported chunk IDs

Plus format checks: 10 candidates per question (fewer only if a contract has fewer chunks), ranks 1..10,
scores non-increasing, chunk text equal to the contract text at the stored offsets.

In [40]:
def read_export(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

check = []
for split, path in export_paths.items():
    recs = read_export(path)
    assert len(recs) == (qa["split"] == split).sum()
    for r in recs:
        cands = r["candidates"]
        assert len(cands) == min(TOP_K, len(chunks_by_contract[r["contract_id"]]))
        assert [c["rank"] for c in cands] == list(range(1, len(cands) + 1))
        assert all(a["bm25_score"] >= b["bm25_score"] for a, b in zip(cands, cands[1:]))
        ctx = contracts[r["contract_id"]]
        assert all(ctx[c["char_start"]:c["char_end"]] == c["text"] for c in cands)
        assert all(c["overlaps_gold"] or not c["contains_gold"] for c in cands)
        if not r["has_answer"]:
            assert not any(c["overlaps_gold"] for c in cands)

    answerable = [r for r in recs if r["has_answer"]]
    hits = [any(c["contains_gold"] for c in r["candidates"]) for r in answerable]
    recalls = []
    for r in answerable:
        ids = {c["chunk_id"] for c in r["candidates"]}
        spans = gold.loc[r["question_id"], "spans"]
        recalls.append(sum(bool(ids & set(s["strict_chunk_ids"])) for s in spans) / len(spans))
    cands = [c for r in recs for c in r["candidates"]]
    check.append({
        "split": split, "questions": len(recs), "answerable": len(answerable),
        "hit@10 (from file)": round(sum(hits) / len(hits), 3),
        "recall@10 (from file)": round(sum(recalls) / len(recalls), 3),
        "positive candidates": sum(c["contains_gold"] for c in cands),
        "partial candidates": sum(c["overlaps_gold"] and not c["contains_gold"] for c in cands),
        "clean negatives": sum(not c["overlaps_gold"] for c in cands),
    })
check = pd.DataFrame(check).set_index("split")

# must match the evaluation above exactly
assert check.loc["test", "recall@10 (from file)"] == round(final["recall_strict@10"].mean(), 3)
assert check.loc["test", "hit@10 (from file)"] == round(final["hit_strict@10"].mean(), 3)
assert check.loc["val", "recall@10 (from file)"] == round(prep_val["3 + stopwords + stemming"]["recall_strict@10"].mean(), 3)
print("All checks passed: the files reproduce the step 4/5 numbers.")
check

All checks passed: the files reproduce the step 4/5 numbers.


,questions,answerable,hit@10 (from file),recall@10 (from file),positive candidates,partial candidates,clean negatives
split,,,,,,,
train,3460,1713,0.918,0.877,2069,568,28313
val,610,287,0.875,0.827,330,105,5095
test,1020,443,0.901,0.858,522,153,8125


In [41]:
# What one record looks like (text shortened for display)
example = read_export(export_paths["val"])[0]
example["candidates"] = [dict(c, text=c["text"][:80] + "...") for c in example["candidates"][:2]]
example["question"] = example["question"][:80] + "..."
print(json.dumps(example, indent=2))

{
  "question_id": "cuad_003__expiration_date",
  "contract_id": "cuad_003",
  "split": "val",
  "category": "Expiration Date",
  "has_answer": true,
  "question": "Highlight the parts (if any) of this contract related to \"Expiration Date\" that ...",
  "query": "Expiration Date On what date will the contract's initial term expire?",
  "candidates": [
    {
      "chunk_id": "cuad_003_c0050",
      "rank": 1,
      "bm25_score": 10.295,
      "text": "term \"Revision\" replaces the German term \"Erzeugnisstand\" (abbreviated \"ES\") and...",
      "char_start": 80582,
      "char_end": 82474,
      "contains_gold": false,
      "overlaps_gold": false
    },
    {
      "chunk_id": "cuad_003_c0051",
      "rank": 2,
      "bm25_score": 9.1435,
      "text": "packaging only)   T   Batch (for packaging only)\n\n\n\n\n\n     2.5 Expiration date  ...",
      "char_start": 82086,
      "char_end": 84108,
      "contains_gold": false,
      "overlaps_gold": false
    }
  ]
}


**How the evidence extraction side uses these files**
1. **End-to-end evaluation (val/test):** run DeBERTa on these 10 chunks per question instead of every chunk.
   The drop from her all-chunks score is what retrieval costs the pipeline.
2. **Hard negatives (train):** candidates with `overlaps_gold == false`. They look relevant to BM25 but contain no
   evidence, so they are harder than random negatives. Keep positives from all chunks as now: if BM25 misses a train
   question's gold chunk, that positive is still worth training on.